# Gold — Camada Analítica (`case.gold`)

Modelo dimensional em estrela: **5 dimensões** e **1 tabela fato**. Timestamps da `tb_chamados_hora` como **fonte de verdade** para tempos de espera e atendimento.

## Ajustes da v2 (sobre o protótipo)

Mantém a estrutura e as decisões do protótipo. O que mudou:

| Ajuste | Por quê |
|---|---|
| `dim_cliente` sem `nome` e `email`; nova coluna `faixa_etaria`; máscara opcional no `id_cliente` | Dados pessoais não servem a nenhum KPI e a Gold vai para CSV. A faixa etária permite testar "o problema é o perfil do usuário" |
| `flag_sem_pesquisa` (que era `nota_media IS NULL`) virou `flag_respondeu` e `flag_nota_valida` | Separa "não respondeu" de "respondeu sem nota válida" (cerca de 6,8 mil chamados) |
| Novas flags: `flag_sem_inicio`, `flag_sem_motivo`, `flag_sem_canal`, `flag_cliente_divergente` | Indício de abandono, junções por texto que não casaram e chamados com cliente diferente entre as duas bases |
| Nova coluna `qtd_pesquisas` | Auditoria dos chamados com mais de uma pesquisa |
| `dim_tempo` com calendário contínuo | Dias sem chamado não somem do calendário |
| Validação com `assert` e sem número fixo (131.838) | O notebook falha se a fato duplicar chamados ou perder chaves |
| Catálogo por widget, `dbutils.notebook.exit` com resumo | Pronto para rodar em Job, como as Silvers |

In [0]:
import json

from pyspark.sql import functions as F


def parametro(nome, padrao):
    try:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)
    except NameError:
        return padrao


catalog = parametro("catalog", "case")
# "sim" aplica hash no id_cliente (use se o id for um CPF). Mantenha o mesmo valor em toda execução.
MASCARAR_ID_CLIENTE = parametro("mascarar_id_cliente", "nao").strip().lower() in ("sim", "true", "1")
SALT_ID_CLIENTE = parametro("salt_id_cliente", "vcredit-rocket-lab")

silver_schema = f"`{catalog}`.silver"
gold_schema = f"`{catalog}`.gold"


def id_cliente_seguro(coluna):
    """id_cliente como está, ou hash SHA-256 com salt. Mesma função na fato e na dimensão = joins preservados."""
    if MASCARAR_ID_CLIENTE:
        return F.sha2(F.concat(F.lit(SALT_ID_CLIENTE), coluna.cast("string")), 256)
    return coluna


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"Schema {gold_schema} garantido. Máscara no id_cliente: {MASCARAR_ID_CLIENTE}")

## Arquitetura dimensional

```
           dim_cliente         dim_atendente
              (47K)               (25)
                 |                    |
  dim_tempo — ft_chamados — dim_canal
   (jan25-jul26)  (131.838)     (4)
                    |             
               dim_motivo        bridge_chamado_pesquisa
                 (14)           (81.021, N:N)
```

### Decisões de ouro

| Decisão | Escolha | Justificativa |
|---|---|---|
| Fonte de verdade para tempos | `tb_chamados_hora` (timestamps) | Campos calculados da tb_chamados divergem em 21% dos casos (PDF: "cabe ao grupo decidir") |
| Granularidade do fato | 1 linha por chamado | Um chamado = um evento de atendimento |
| Bridge de pesquisa | N:N chamado ↔ pesquisa | 802 chamados têm múltiplas pesquisas; preservar para auditoria |
| Idade inválida | NULL quando `status_idade ≠ ok` | Evita distorcer análises demográficas |
| Chaves | Naturais (id_cliente, id_motivo, etc.) | Dimensões pequenas, sem SCD |

### Estratégia de joins na fato

| Tabela | Tipo | Motivo |
|---|---|---|
| tb_chamados_hora | INNER | Base 1:1, mesma origem |
| tb_motivos | LEFT (via `nome_motivo_normalizado`) | FK `id_motivo` |
| tb_canais | LEFT (via `nome_canal`) | FK `id_canal` |
| tb_clientes | LEFT | Validar FK `id_cliente` (2.363 sem cliente) |
| tb_custos | LEFT | Medida `custo` (1.550 sem custo) |
| tb_pesquisa_satisfacao_chamado | LEFT | Medida `nota_atendimento` (58.455 sem nota útil) |
| tb_base_atendentes | LEFT | Validar FK `id_atendente` (764 com -1) |

**Idempotência:** todas as gravações usam `mode("overwrite")` + `overwriteSchema`.

In [0]:
# --- dim_cliente ---
# Sem nome e e-mail: dados pessoais que nenhum KPI usa e que iriam para o CSV do Power BI.
# Idade NULL quando status_idade ≠ "ok" (455 nulos + 244 zero + 131 >120 na origem).
idade = F.when(F.col("status_idade") == "ok", F.col("idade"))
df_dim_cliente = spark.table(f"{silver_schema}.tb_clientes").select(
    id_cliente_seguro(F.col("id_cliente")).alias("id_cliente"),
    F.col("regiao"),
    idade.alias("idade"),
    F.when(idade.isNull(), "sem informacao")
    .when(idade < 30, "18-29")
    .when(idade < 45, "30-44")
    .when(idade < 60, "45-59")
    .otherwise("60+").alias("faixa_etaria"),
    F.col("data_cadastro"),
    F.col("dia_vencimento_fatura"),
    F.col("status_idade"),
)

df_dim_cliente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_cliente")
print(f"dim_cliente: {df_dim_cliente.count():,} linhas")

In [0]:
# --- dim_atendente ---
df_dim_atendente = spark.table(f"{silver_schema}.tb_base_atendentes")

df_dim_atendente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_atendente")
print(f"dim_atendente: {df_dim_atendente.count()} linhas")

In [0]:
# --- dim_canal ---
df_dim_canal = spark.table(f"{silver_schema}.tb_canais")

df_dim_canal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_canal")
print(f"dim_canal: {df_dim_canal.count()} linhas")

In [0]:
# --- dim_motivo ---
df_dim_motivo = spark.table(f"{silver_schema}.tb_motivos")

df_dim_motivo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_motivo")
print(f"dim_motivo: {df_dim_motivo.count()} linhas")

In [0]:
# --- dim_tempo ---
# Calendário contínuo do primeiro ao último dia com chamado (dias sem chamado também existem).
limites = spark.table(f"{silver_schema}.tb_chamados_hora").select(
    F.min(F.to_date(F.col("hora_abertura_chamado"))).alias("ini"),
    F.max(F.to_date(F.col("hora_abertura_chamado"))).alias("fim"),
)
df_data = limites.select(F.explode(F.sequence("ini", "fim")).alias("data"))

df_dim_tempo = df_data.select(
    F.col("data"),
    F.year("data").alias("ano"),
    F.month("data").alias("mes_num"),
    F.date_format("data", "MMMM").alias("mes_nome"),
    F.dayofmonth("data").alias("dia"),
    F.dayofweek("data").alias("dia_semana_num"),
    F.date_format("data", "EEEE").alias("dia_semana_nome"),
    F.date_format("data", "yyyy-MM").alias("mes_ano"),
    F.quarter("data").alias("trimestre"),
    F.when(F.month("data") <= 6, 1).otherwise(2).alias("semestre"),
)

df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_tempo")
print(f"dim_tempo: {df_dim_tempo.count()} linhas")

In [0]:
# --- Leitura das tabelas Silver ---
df_chamados = spark.table(f"{silver_schema}.tb_chamados")
df_chamados_hora = spark.table(f"{silver_schema}.tb_chamados_hora")
df_clientes = spark.table(f"{silver_schema}.tb_clientes")
df_custos = spark.table(f"{silver_schema}.tb_custos")
df_pesquisa = spark.table(f"{silver_schema}.tb_pesquisa_satisfacao_chamado")
df_motivos = spark.table(f"{silver_schema}.tb_motivos")
df_canais = spark.table(f"{silver_schema}.tb_canais")
df_atendentes = spark.table(f"{silver_schema}.tb_base_atendentes")

# Flag extra só se a Silver de horários trouxer a coluna de integridade do cliente
extras = []
if "status_integridade_cliente" in df_chamados_hora.columns:
    extras.append((F.col("ch.status_integridade_cliente") == "CLIENTE_DIVERGENTE").alias("flag_cliente_divergente"))

# --- Build da tabela fato ft_chamados ---
# Decisão de ouro: tb_chamados_hora é a fonte de verdade para timestamps.
# tempo_espera e tempo_atendimento são recalculados a partir dos timestamps.
df_gold = (
    df_chamados.alias("c")
    .join(df_chamados_hora.alias("ch"), on="id_chamado", how="inner")
    .join(df_motivos.alias("m"),
          on=F.col("c.motivo") == F.col("m.nome_motivo"), how="left")
    .join(df_canais.alias("can"),
          on=F.col("c.canal") == F.col("can.nome_canal"), how="left")
    .join(df_clientes.withColumn("_tem_cliente", F.lit(True)).alias("cli"), on="id_cliente", how="left")
    .join(df_custos.alias("cu"), on="id_chamado", how="left")
    .join(df_pesquisa.alias("ps"), on="id_chamado", how="left")
    .join(df_atendentes.alias("a"),
          on=F.col("c.id_atendente") == F.col("a.id_atendente"), how="left")
    .select(
        F.col("c.id_chamado"),
        id_cliente_seguro(F.col("c.id_cliente")).alias("id_cliente"),
        F.col("c.motivo"),
        F.col("c.canal"),
        F.col("c.resolvido"),
        F.col("c.id_atendente"),
        # FKs para dimensoes
        F.col("m.id_motivo"),   # FK dim_motivo
        F.col("can.id_canal"),  # FK dim_canal
        # Timestamps (fonte de verdade: tb_chamados_hora)
        F.col("ch.hora_abertura_chamado"),
        F.col("ch.hora_inicio_atendimento"),
        F.col("ch.hora_finalizacao_atendimento"),
        F.col("ch.status_consistencia"),
        # Tempos calculados a partir dos timestamps (fonte de verdade)
        F.when(
            (F.col("ch.status_consistencia") == "CONSISTENTE")
            & F.col("ch.hora_inicio_atendimento").isNotNull()
            & F.col("ch.hora_abertura_chamado").isNotNull(),
            (F.unix_timestamp(F.col("ch.hora_inicio_atendimento"))
             - F.unix_timestamp(F.col("ch.hora_abertura_chamado")))
        ).cast("int").alias("tempo_espera"),
        F.when(
            (F.col("ch.status_consistencia") == "CONSISTENTE")
            & F.col("ch.hora_finalizacao_atendimento").isNotNull()
            & F.col("ch.hora_inicio_atendimento").isNotNull(),
            (F.unix_timestamp(F.col("ch.hora_finalizacao_atendimento"))
             - F.unix_timestamp(F.col("ch.hora_inicio_atendimento")))
        ).cast("int").alias("tempo_atendimento"),
        # Tempos originais (para auditoria/comparacao)
        F.col("c.tempo_espera").alias("tempo_espera_original"),
        F.col("c.tempo_atendimento").alias("tempo_atendimento_original"),
        # Dimensoes derivadas de data
        F.to_date(F.col("ch.hora_abertura_chamado")).alias("data_abertura"),
        F.date_format(F.col("ch.hora_abertura_chamado"), "yyyy-MM").alias("mes_abertura"),
        F.year(F.col("ch.hora_abertura_chamado")).alias("ano_abertura"),
        F.month(F.col("ch.hora_abertura_chamado")).alias("mes_num"),
        F.dayofweek(F.col("ch.hora_abertura_chamado")).alias("dia_semana_num"),
        F.date_format(F.col("ch.hora_abertura_chamado"), "EEEE").alias("dia_semana"),
        F.hour(F.col("ch.hora_abertura_chamado")).alias("hora_abertura"),
        # Fatos
        F.col("cu.custo"),
        F.col("ps.nota_media").alias("nota_atendimento"),
        F.coalesce(F.col("ps.qtd_pesquisas"), F.lit(0)).alias("qtd_pesquisas"),
        # Flags de qualidade
        F.col("cli._tem_cliente").isNull().alias("flag_sem_cliente"),  # cliente não existe na Silver (nome nulo não conta)
        F.col("cu.custo").isNull().alias("flag_sem_custo"),
        F.col("ps.qtd_pesquisas").isNotNull().alias("flag_respondeu"),     # tem alguma pesquisa
        F.col("ps.nota_media").isNotNull().alias("flag_nota_valida"),      # tem nota válida (1 a 5)
        (F.col("c.id_atendente") == -1).alias("flag_sem_atendente"),
        (F.col("ch.status_consistencia") != "CONSISTENTE").alias("flag_ordem_invalida"),
        F.col("ch.hora_inicio_atendimento").isNull().alias("flag_sem_inicio"),  # indício de abandono
        F.col("m.id_motivo").isNull().alias("flag_sem_motivo"),
        F.col("can.id_canal").isNull().alias("flag_sem_canal"),
        *extras,
    )
)

# Gravacao idempotente
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.ft_chamados")
)

print(f"Tabela {gold_schema}.ft_chamados criada/atualizada!")
print(f"Total de linhas: {spark.table(f'{gold_schema}.ft_chamados').count():,}")

## Bridge table — `bridge_chamado_pesquisa`

Preserva a relação **N:N** entre chamados e pesquisas de satisfação. 802 chamados têm múltiplas pesquisas; a tabela fato aggrega (`nota_media`), mas a bridge mantém as notas individuais para auditoria e análise de divergências.

In [0]:
# --- bridge_chamado_pesquisa ---
# N:N entre ft_chamados e pesquisas individuais (81.021 pesquisas, 802 chamados com múltiplas).
df_bridge = spark.table(f"{silver_schema}.tb_pesquisa_satisfacao").select(
    F.col("id_pesquisa"),
    F.col("id_chamado"),
    F.col("nota_atendimento"),
    F.col("nota_original"),
    F.col("flag_sem_nota"),
    F.col("flag_nota_fora_dominio"),
    F.col("flag_chamado_multiplas_pesquisas"),
)

df_bridge.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_chamado_pesquisa")
print(f"bridge_chamado_pesquisa: {df_bridge.count():,} linhas")

In [0]:
# === Validacao (falha o notebook se algo estiver errado) ===
erros = []
avisos = []

print("=== Dimensões ===")
for dim in ["dim_cliente", "dim_atendente", "dim_canal", "dim_motivo", "dim_tempo"]:
    print(f"  {dim}: {spark.table(f'{gold_schema}.{dim}').count():,} linhas")

fato = spark.table(f"{gold_schema}.ft_chamados")
n_fato = fato.count()
esperado = (
    spark.table(f"{silver_schema}.tb_chamados").select("id_chamado").distinct()
    .join(spark.table(f"{silver_schema}.tb_chamados_hora").select("id_chamado").distinct(), "id_chamado")
    .count()
)
print(f"\n=== Fato ===\n  ft_chamados: {n_fato:,} linhas (esperado pelos Silvers: {esperado:,})")
if n_fato != esperado:
    erros.append(f"ft_chamados tem {n_fato} linhas, esperado {esperado} (junção duplicou ou perdeu chamados)")
n_ids = fato.select("id_chamado").distinct().count()
if n_ids != n_fato:
    erros.append(f"ft_chamados tem id_chamado repetido ({n_fato} linhas, {n_ids} ids)")

print("\n  Flags de qualidade:")
flags = [c for c in fato.columns if c.startswith("flag_")]
resumo_flags = fato.select([F.sum(F.col(c).cast("int")).alias(c) for c in flags]).collect()[0].asDict()
for c in flags:
    print(f"    {c}: {resumo_flags[c]:,}")

print("\n  Tempos (fonte de verdade = timestamps):")
fato.select(
    F.sum(F.col("tempo_espera").isNull().cast("int")).alias("espera_null"),
    F.sum(F.col("tempo_espera_original").isNull().cast("int")).alias("espera_orig_null"),
    F.sum(F.col("tempo_atendimento").isNull().cast("int")).alias("atendimento_null"),
    F.sum(F.col("tempo_atendimento_original").isNull().cast("int")).alias("atend_orig_null"),
).show()

print("=== Bridge ===")
df_bridge = spark.table(f"{gold_schema}.bridge_chamado_pesquisa")
print(f"  bridge_chamado_pesquisa: {df_bridge.count():,} linhas")
df_bridge.groupBy("flag_chamado_multiplas_pesquisas").count().show()
orfas_bridge = df_bridge.select("id_chamado").distinct().join(fato.select("id_chamado"), "id_chamado", "left_anti").count()
print(f"  pesquisas de chamado inexistente na fato: {orfas_bridge:,}")
if orfas_bridge:
    avisos.append(f"{orfas_bridge} chamados da bridge não existem na fato")

print("=== Integridade FKs (órfãs = chave preenchida que não existe na dimensão) ===")


def orfas(coluna_fato, dim, coluna_dim):
    d = spark.table(f"{gold_schema}.{dim}").select(F.col(coluna_dim).alias(coluna_fato)).distinct()
    return fato.filter(F.col(coluna_fato).isNotNull()).select(coluna_fato).distinct().join(d, coluna_fato, "left_anti").count()


for coluna, dim, col_dim in [("id_motivo", "dim_motivo", "id_motivo"), ("id_canal", "dim_canal", "id_canal"),
                             ("data_abertura", "dim_tempo", "data")]:
    n = orfas(coluna, dim, col_dim)
    print(f"  {coluna} órfãos em {dim}: {n}")
    if n:
        erros.append(f"{coluna}: {n} valores sem correspondência em {dim}")

# cliente: a fato mantém o chamado sem cliente (flag); precisa bater com a dimensão
sem_cli_dim = fato.join(
    spark.table(f"{gold_schema}.dim_cliente").select("id_cliente").distinct().withColumn("_ok", F.lit(1)),
    "id_cliente", "left").filter(F.col("_ok").isNull()).count()
print(f"  chamados cujo cliente não está na dim_cliente: {sem_cli_dim:,} (flag_sem_cliente = {resumo_flags['flag_sem_cliente']:,})")
if sem_cli_dim != resumo_flags["flag_sem_cliente"]:
    erros.append("flag_sem_cliente não bate com os chamados sem cliente na dim_cliente")

print("\n=== Avisos (não bloqueiam) ===")
for nome in ["flag_sem_motivo", "flag_sem_canal"]:
    if resumo_flags.get(nome):
        avisos.append(f"{resumo_flags[nome]} chamados com {nome}: a junção por nome não casou")
print("  " + ("\n  ".join(avisos) if avisos else "nenhum"))

if erros:
    raise ValueError("Validação da Gold falhou: " + " | ".join(erros))

resumo = {"ft_chamados": int(n_fato), "flags": {k: int(v or 0) for k, v in resumo_flags.items()}, "avisos": avisos}
print(json.dumps(resumo, ensure_ascii=False))
try:
    dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False))
except NameError:
    pass